In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score


# 1. CARREGAR
df = pd.read_excel(r'../data/dataset_games.xlsx')

# Criando a feature de "Titãs de Bilhões" (Rockstar + Hype >= 9)
df['Is_Main_Franchise_Titan'] = np.where((df['Is_Rockstar_Game'] == 1) & (df['Announcement_Hype_Score'] >= 9), 1, 0)

features = [
    'Metacritic_Score', 
    'Market_Size_USD_B', 
    'Has_Live_Service',
    'Consoles_Active_Base (M)', 
    'Announcement_Hype_Score', 
    'Is_Main_Franchise_Titan',
    'Trailer_Views_First_24h (M)',          
    'Years_Since_Last_Franchise_Title',     
    'Digital_Share_Percentage',             
    'Social_Media_Followers_Pre_Launch (M)' 
]

# --- Target Baseado em Proporção ---
# Taxa de Penetração = Que percentagem da base de consolas comprou o jogo no Ano 1
df['Market_Penetration_Rate'] = df['Year_1_Sales_Est (M)'] / df['Consoles_Active_Base (M)']
target = 'Market_Penetration_Rate'

coluna_nome = 'Game_Title' if 'Game_Title' in df.columns else 'Game'
df_treino = df[df[coluna_nome] != 'Grand Theft Auto VI'].dropna(subset=[target])

X = df_treino[features]
y = df_treino[target]  # O Alvo agora é a taxa (ex: 0.15 significa 15% de penetração)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



# 2. TREINAR E AVALIAR O XGBOOST (TAXA DE PENETRAÇÃO)
modelo_xgb = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
modelo_xgb.fit(X_train, y_train)

y_pred_xgb = modelo_xgb.predict(X_test)

# Avaliação das métricas baseada diretamente na taxa de penetração
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print("--- Avaliação do Modelo (XGBoost Regressor via Penetração) ---")
print(f"Erro Médio Absoluto na Taxa: {mae_xgb*100:.2f}% de desvio na base")
print(f"Coeficiente de Determinação (R²): {r2_xgb:.2f}")
print("-" * 62)


# 3. TREINAR E AVALIAR A REGRESSÃO LINEAR (TAXA DE PENETRAÇÃO)
modelo_linear = LinearRegression()
modelo_linear.fit(X_train, y_train)

y_pred_linear = modelo_linear.predict(X_test)
mae_linear = mean_absolute_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)

print("\n--- Avaliação do Modelo (Regressão Linear via Penetração) ---")
print(f"Erro Médio Absoluto na Taxa: {mae_linear*100:.2f}% de desvio na base")
print(f"Coeficiente de Determinação (R²): {r2_linear:.2f}")
print("-" * 61)


# 4. PREDIÇÃO DO CENÁRIO DO GTA 6
base_consolas_gta6 = 128.0  

cenario_gta_6 = pd.DataFrame([{
    'Metacritic_Score': 98,
    'Market_Size_USD_B': 230.0,             
    'Has_Live_Service': 1,
    'Consoles_Active_Base (M)': base_consolas_gta6,      
    'Announcement_Hype_Score': 10,
    'Is_Main_Franchise_Titan': 1,
    'Trailer_Views_First_24h (M)': 475.0,   
    'Years_Since_Last_Franchise_Title': 13, 
    'Digital_Share_Percentage': 75.0,       
    'Social_Media_Followers_Pre_Launch (M)': 65.0   
}])

cenario_gta_6 = cenario_gta_6[features]

# O modelo vai prever a taxa de conversão esperada
taxa_prevista_xgb = modelo_xgb.predict(cenario_gta_6)[0]
taxa_prevista_linear = modelo_linear.predict(cenario_gta_6)[0]

# Cliper aplicado diretamente nas taxas para evitar absurdos negativos
taxa_prevista_linear_clipped = np.clip(taxa_prevista_linear, 0, None)

# --- A MÁGICA DO NEGÓCIO: Convertendo taxa prevista em cópias reais ---
vendas_reais_xgb = taxa_prevista_xgb * base_consolas_gta6
vendas_reais_linear = taxa_prevista_linear_clipped * base_consolas_gta6

print("\n📈 --- TAXAS DE PENETRAÇÃO PREVISTAS ---")
print(f"XGBoost estima que o jogo será comprado por : {taxa_prevista_xgb*100:.2f}% da base de consolas.")
print(f"Regressão Linear estima uma taxa de        : {taxa_prevista_linear_clipped*100:.2f}% da base de consolas.")

print("\n🚀 --- COMPARATIVO FINAL EM CÓPIAS VENDIDAS (ANO 1) ---")
print(f"XGBoost Regressor  : {vendas_reais_xgb:.2f} milhões de cópias.")
print(f"Regressão Linear   : {vendas_reais_linear:.2f} milhões de cópias.")
print("-" * 54)

--- Avaliação do Modelo (XGBoost Regressor via Penetração) ---
Erro Médio Absoluto na Taxa: 5.31% de desvio na base
Coeficiente de Determinação (R²): 0.68
--------------------------------------------------------------

--- Avaliação do Modelo (Regressão Linear via Penetração) ---
Erro Médio Absoluto na Taxa: 9.15% de desvio na base
Coeficiente de Determinação (R²): 0.23
-------------------------------------------------------------

📈 --- TAXAS DE PENETRAÇÃO PREVISTAS ---
XGBoost estima que o jogo será comprado por : 28.88% da base de consolas.
Regressão Linear estima uma taxa de        : 0.00% da base de consolas.

🚀 --- COMPARATIVO FINAL EM CÓPIAS VENDIDAS (ANO 1) ---
XGBoost Regressor  : 36.97 milhões de cópias.
Regressão Linear   : 0.00 milhões de cópias.
------------------------------------------------------
